# Figure: Supervised eval — train/valid learning curves + gene-level score distribution

This figure summarizes how the supervised track-prediction model is trained and how well it scores genes. **Part A** plots aggregated (mean ± std across the 8 cross-validation folds) training and validation learning curves — loss, Pearson's R, and R² vs. number of training batches — comparing the LM-finetuned model (**Shorkie**, `self_supervised_*`) against the from-scratch baseline (**Shorkie_Random_Init**, `supervised_*`). **Part B** plots the gene-level prediction-quality distribution (Pearson's R density over genes) for the two models on RNA-seq tracks, averaged across folds.

**Reproduces:** the supervised cross-validation learning curves and the gene-level Pearson's-R density panel of the track-prediction evaluation figure.

**Upstream:** the per-fold training logs (`train/f{i}c0/train.out`) are produced by the 8-fold supervised training under `scripts/02_train/` (`westminster_train_folds.py`); the gene-level eval tables (`gene_level_eval_rc/f{i}c0/<track>/gene_acc.txt`) are produced by `scripts/03_eval/supervised/track_prediction_eval/2_bin_gene_level_metrics/2_yeast_rna_seq_models.py`. Both live under `results.train_logs` (the supervised `16bp` experiment root) and are work-dir-only intermediates, not part of the released data manifest.

**Requires:** the `yeast_ml` conda env with this repo installed (`pip install -e .`). CPU only — this notebook just parses text logs / TSVs and plots; no GPU and no model inference.

**Source script:** ported from `scripts/03_eval/supervised/track_prediction_eval/1_train_valid_curves/1_compare_cross_valid_curves.py` (Part A, aggregated two-group curves) and `scripts/03_eval/supervised/track_prediction_eval/2_bin_gene_level_metrics/3_gene_level_score_dist_viz.py` (Part B, gene-level Pearson's-R density).

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from shorkie import config

## Resolve paths from config

All inputs live under `results.train_logs`, which resolves to the supervised `16bp` experiment root. Under it the two models are laid out as:

- **Shorkie** (LM-finetuned): `self_supervised_unet_small_bert_drop/...`
- **Shorkie_Random_Init** (from scratch): `supervised_unet_small_bert_drop_variants/learning_rate_0.0005/...`

with training logs in `train/f{fold}c0/train.out` and gene-level eval tables in `gene_level_eval_rc/f{fold}c0/<track_type>/gene_acc.txt`.

In [ ]:
train_logs_root = config.path("results.train_logs")
print("train_logs root:", train_logs_root)

NUM_FOLDS = config.get("models.num_folds", 8)
EVAL_SUBDIR = "gene_level_eval_rc"   # the eval experiment whose gene_acc.txt we read

# Per-model subpath prefixes (relative to train_logs_root)
MODEL_SUBDIR = {
    "Shorkie":            "self_supervised_unet_small_bert_drop",
    "Shorkie_Random_Init": "supervised_unet_small_bert_drop_variants/learning_rate_0.0005",
}
# train.out lives under the plain model dir (no learning_rate suffix for the scratch baseline)
TRAIN_SUBDIR = {
    "Shorkie":            "self_supervised_unet_small_bert_drop",
    "Shorkie_Random_Init": "supervised_unet_small_bert_drop_variants/learning_rate_0.0005",
}

for name, sub in MODEL_SUBDIR.items():
    print(name, "->", os.path.join(str(train_logs_root), sub))

## Part A — parse per-fold training logs

Each `train.out` line of the form

```
Epoch N - 76s - train_loss: ... - train_r: ... - train_r2: ... - valid_loss: ... - valid_r: ... - valid_r2: ...
```

is split on `" - "` and the six metrics extracted (matching the parser in `1_compare_cross_valid_curves.py`).

In [ ]:
def parse_train_out(path):
    """Return dict of metric-name -> 1D np.array over epochs, parsing 'Epoch ...' lines."""
    keys = ["train_loss", "train_r", "train_r2", "valid_loss", "valid_r", "valid_r2"]
    out = {k: [] for k in keys}
    with open(path, "rt") as fh:
        for line in fh:
            line = line.strip()
            if not line.startswith("Epoch "):
                continue
            parts = line.split(" - ")
            # parts[0]='Epoch N', parts[1]='<t>s', parts[2:8] are the six metrics
            try:
                vals = [float(parts[2 + i].split(": ")[1]) for i in range(6)]
            except (IndexError, ValueError):
                continue
            for k, v in zip(keys, vals):
                out[k].append(v)
    return {k: np.asarray(v) for k, v in out.items()}


def load_model_logs(model_name):
    """Parse all folds for one model; return dict metric -> list of per-fold arrays."""
    base = os.path.join(str(train_logs_root), TRAIN_SUBDIR[model_name], "train")
    keys = ["train_loss", "train_r", "train_r2", "valid_loss", "valid_r", "valid_r2"]
    per_metric = {k: [] for k in keys}
    for fold in range(NUM_FOLDS):
        fp = os.path.join(base, f"f{fold}c0", "train.out")
        if not os.path.exists(fp):
            print(f"[skip] missing {fp}")
            continue
        d = parse_train_out(fp)
        if len(d["train_loss"]) == 0:
            print(f"[skip] no Epoch lines in {fp}")
            continue
        for k in keys:
            per_metric[k].append(d[k])
    return per_metric


logs = {name: load_model_logs(name) for name in MODEL_SUBDIR}
for name, d in logs.items():
    print(name, "folds parsed:", len(d["valid_r"]))

## Part A — plot aggregated train/valid curves (mean ± std across folds)

For each metric we smooth each fold with a small moving average, align all folds to the shortest length, then plot the across-fold mean with a ±std band (mirroring `plot_aggregated_two_groups` in the source). `STEPS_PER_EPOCH` converts the epoch axis to training batches.

In [ ]:
STEPS_PER_EPOCH = 500   # steps_per_epoch_max from the training params
WINDOW = 11             # moving-average window (source default)
TRIM_END = 5            # drop the last few noisy epochs (source default)


def moving_average(x, window=WINDOW, trim_end=TRIM_END):
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    n = len(x)
    for j in range(n):
        w = min(j, n - 1 - j, window // 2)
        out[j] = x[max(j - w, 0): j + w + 1].mean()
    return out[:-trim_end] if trim_end > 0 else out


def aggregate(fold_arrays):
    """Smooth each fold, align to shortest length, return (mean, std, length)."""
    sm = [moving_average(a) for a in fold_arrays]
    L = min(len(s) for s in sm)
    M = np.vstack([s[:L] for s in sm])
    return M.mean(axis=0), M.std(axis=0), L


metric_panels = [
    ("loss", "Loss"),
    ("r", "Pearson's R"),
    ("r2", "R²"),
]
split_titles = {"train": "Training", "valid": "Validation"}
group_styles = {"Shorkie": dict(linestyle="-"), "Shorkie_Random_Init": dict(linestyle="--")}

fig, axes = plt.subplots(2, 3, figsize=(16, 9), dpi=120)
for row, split in enumerate(["train", "valid"]):
    for col, (mkey, mlabel) in enumerate(metric_panels):
        ax = axes[row, col]
        for name in MODEL_SUBDIR:
            arrays = logs[name][f"{split}_{mkey}"]
            if not arrays:
                continue
            mean, std, L = aggregate(arrays)
            x = (np.arange(L) + 1) * STEPS_PER_EPOCH
            ax.plot(x, mean, linewidth=2,
                    label=f"{name} (final={mean[-1]:.3f}\u00b1{std[-1]:.3f})",
                    **group_styles[name])
            ax.fill_between(x, mean - std, mean + std, alpha=0.2)
        ax.set_xlabel("# Training Batches")
        ax.set_ylabel(f"{split_titles[split]} {mlabel}")
        ax.set_title(f"{split_titles[split]} {mlabel}")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
fig.suptitle("Supervised cross-validation learning curves (8 folds, mean ± std)", fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## Part B — load per-fold gene-level eval tables and average across folds

For each model and RNA-seq track type we read `gene_acc.txt` for every fold, concatenate, then average each metric per gene over folds (mirroring `load_and_average_across_folds` in `3_gene_level_score_dist_viz.py`). We keep the representative RNA-seq track types used for the gene-level panel.

In [ ]:
TRACK_TYPES = ["RNA-Seq", "1000-RNA-seq"]
GENE_METRIC = "pearsonr"   # the representative gene-level metric we display


def load_gene_acc(model_name, track_type):
    """Average per-gene metrics across folds for one model/track_type. Returns DataFrame."""
    base = os.path.join(str(train_logs_root), MODEL_SUBDIR[model_name], EVAL_SUBDIR)
    frames = []
    for fold in range(NUM_FOLDS):
        fp = os.path.join(base, f"f{fold}c0", track_type, "gene_acc.txt")
        if not os.path.exists(fp):
            print(f"[skip] missing {fp}")
            continue
        frames.append(pd.read_csv(fp, sep="\t"))
    if not frames:
        return pd.DataFrame()
    allf = pd.concat(frames, ignore_index=True)
    return allf.groupby("gene_id", as_index=False).mean(numeric_only=True)


gene_tables = {
    name: {tt: load_gene_acc(name, tt) for tt in TRACK_TYPES}
    for name in MODEL_SUBDIR
}
for name in MODEL_SUBDIR:
    for tt in TRACK_TYPES:
        df = gene_tables[name][tt]
        print(f"{name:20s} {tt:14s} genes={len(df)}")

## Part B — gene-level Pearson's-R density

Gaussian-KDE density of per-gene Pearson's R over [0, 1] for each model and RNA-seq track type (faithful to the combined-density panel in the source). Shorkie is drawn in the saturated color, Shorkie_Random_Init in a lightened tint of the same hue.

In [ ]:
base_color = {"RNA-Seq": "#d62728", "1000-RNA-seq": "#2ca02c"}


def lighten(color, amount=0.6):
    import matplotlib.colors as mc
    c = np.array(mc.to_rgb(color))
    return (1 - amount) * c + amount


fig, ax = plt.subplots(figsize=(7, 4.2), dpi=120)
xg = np.linspace(0, 1, 200)
for tt in TRACK_TYPES:
    for name in MODEL_SUBDIR:
        df = gene_tables[name][tt]
        if df.empty or GENE_METRIC not in df.columns:
            continue
        vals = df[GENE_METRIC].dropna().values
        if vals.size < 2:
            continue
        yg = gaussian_kde(vals)(xg)
        color = base_color[tt] if name == "Shorkie" else lighten(base_color[tt])
        ax.plot(xg, yg, color=color, label=f"{tt} ({name})")
        ax.fill_between(xg, 0, yg, color=color, alpha=0.2)
ax.set_title("Pearson's R density of gene-level predictions")
ax.set_xlabel("Pearson's R")
ax.set_ylabel("Density")
ax.set_xlim(0, 1)
ax.grid(axis="y", alpha=0.5)
ax.legend(loc="upper left", frameon=False, fontsize=9)
fig.tight_layout()
plt.show()